# Entity ***Positions***
+ Layer **silver**
+ Priority **2**
---
### Imports

In [0]:
%run ../common/medallion_functions

### Parameters

In [0]:
# layer = dbutils.widgets.get("layer")
# entity_name = dbutils.widgets.get("entity_name")
# load_pattern = dbutils.widgets.get("load_pattern")
layer = "silver"
entity_name="positions"
load_pattern = "1"

### Execution

In [0]:
bz_positions_df = spark.read.table(f"uniswap.bronze.{entity_name}")

In [0]:
sv_positions_df = spark.sql(f"""
    WITH cte_deduplicate_bz AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY id
                ORDER BY _load_timestamp_bronze DESC
            ) AS rn
        FROM {{df}}
    )
    SELECT
        id AS pk_position_id,
        transaction.id AS tx_id,
        pool.id AS pool_id,
        token0.id AS token0_id,
        token1.id AS token1_id,
        tickLower.id AS tick_lower_id,
        tickUpper.id AS tick_upper_id,
        owner AS owner_address,
        CAST(liquidity AS DOUBLE) AS liquidity,
        CAST(depositedToken0 AS DOUBLE) AS deposited_token0,
        CAST(depositedToken1 AS DOUBLE) AS deposited_token1,
        CAST(withdrawnToken0 AS DOUBLE) AS withdrawn_token0,
        CAST(withdrawnToken1 AS DOUBLE) AS withdrawn_token1,
        CAST(collectedFeesToken0 AS DOUBLE) AS collected_fees_token0,
        CAST(collectedFeesToken1 AS DOUBLE) AS collected_fees_token1,
        CAST(FROM_UNIXTIME(CAST(transaction.timestamp AS INT)) AS TIMESTAMP) AS tx_timestamp,
        CURRENT_TIMESTAMP() AS _load_timestamp_silver
    FROM cte_deduplicate_bz
    WHERE rn = 1
""", df = bz_positions_df)

### Save & exit

In [0]:
load_result = load_entity(sv_positions_df,
                        entity_name,
                        load_pattern,
                        layer
                        )

In [0]:
dbutils.notebook.exit(load_result)